# Customer Analytics & Purchase Pattern Analysis
Explore 5,000 synthetic e-commerce transactions using Python. All results are illustrative, not actual company performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent

df = pd.read_csv(ROOT / 'data' / 'transactions.csv', parse_dates=['order_date'])
df.head()

## 1. Data quality and overview

In [ ]:
print('Shape:', df.shape)
print('Missing values:\n', df.isna().sum())
print('Duplicate order IDs:', df['order_id'].duplicated().sum())
print(df.describe(include='all').T)

## 2. Executive KPIs

In [ ]:
delivered = df[df.order_status.eq('Delivered')].copy()
kpis = pd.Series({
    'Total transactions': len(df),
    'Delivered orders': len(delivered),
    'Purchasing customers': delivered.customer_id.nunique(),
    'Net revenue': df.net_revenue.sum(),
    'AOV per delivered order': df.net_revenue.sum() / max(len(delivered), 1)
})
kpis

## 3. Monthly revenue trend

In [ ]:
monthly = df.set_index('order_date').resample('MS').net_revenue.sum()
monthly.plot(figsize=(10,4), marker='o', title='Monthly Net Revenue')
plt.ylabel('Net revenue'); plt.xlabel('Month'); plt.grid(alpha=.2); plt.show()

## 4. Category and product performance

In [ ]:
cat = df.groupby('category', as_index=False).net_revenue.sum().sort_values('net_revenue', ascending=False)
sns.barplot(data=cat, x='net_revenue', y='category')
plt.title('Net Revenue by Category'); plt.show()

product = df.groupby(['category','product_name'], as_index=False).agg(
    units_sold=('quantity','sum'), net_revenue=('net_revenue','sum')
).sort_values('net_revenue', ascending=False)
product.head(10)

## 5. Repeat purchase behaviour

In [ ]:
freq = delivered.groupby('customer_id').order_id.nunique()
repeat_pct = (freq.gt(1).mean() * 100) if len(freq) else 0
print(f'Repeat customer rate: {repeat_pct:.1f}%')
print(freq.value_counts().sort_index().head(10))

## 6. RFM-style customer segmentation

In [ ]:
snapshot = df.order_date.max() + pd.Timedelta(days=1)
rfm = delivered.groupby('customer_id').agg(
    recency=('order_date', lambda x: (snapshot - x.max()).days),
    frequency=('order_id','nunique'),
    monetary=('net_revenue','sum')
)
rfm['value_segment'] = pd.qcut(rfm.monetary.rank(method='first'), 4,
                               labels=['Low value','Mid value','High value','Top value'])
rfm.groupby('value_segment').agg(customers=('monetary','size'), avg_monetary=('monetary','mean'))

## 7. Business takeaways and next steps
Interpret the computed outputs after running the notebook. Investigate category concentration, repeat purchase rate, customer value distribution and month-to-month changes. Validate all KPIs against SQL and Power BI before publishing. Do not invent business impact from synthetic data.